# Questão 01: Pré-Treinamento Continuado (DOMPI-2025)

Adaptação contínua do **Qwen2.5-0.5B** ao corpus do Diário Oficial dos Municípios do Piauí. Este notebook contém o *setup* compartilhado (Seção 0) seguido da Questão 01. As saídas preservadas são as da execução validada no Kaggle (GPU T4).


## Seção 0: Setup global

A célula abaixo precisa ser a primeira célula de código do notebook: `CUDA_VISIBLE_DEVICES` limita a sessão a uma única GPU antes de qualquer `import torch`. Com as duas T4 visíveis, o Trainer ativaria DataParallel e o gather dos logits (vocabulário de 151k tokens) estoura a memória da GPU 0. O modelo de 0.5B cabe com folga em uma única T4.

In [ ]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

import time
import random

MODO_TESTE = False
RODAR_Q1 = True
RODAR_BONUS_15B = True

SEED = 42
MAX_LEN = 256
MODEL_NAME = 'Qwen/Qwen2.5-0.5B'
MODEL_NAME_BONUS = 'Qwen/Qwen2.5-1.5B'
T0 = time.time()

LIMITE_BONUS_H = 9.5
LIMITE_TETO_H = 11.0

if MODO_TESTE:
    Q1_MAX_DOCS = 300
    Q1_N_AVALIACAO = 50
    Q1_EPOCAS = 1
    Q2_ALVO_PARES = 120
    Q2_EPOCAS = 1
    Q3_EPOCAS = 1
    BONUS_MAX_STEPS = 10
    N_BENCHMARK_Q1 = 5
else:
    Q1_MAX_DOCS = None
    Q1_N_AVALIACAO = 1000
    Q1_EPOCAS = 2
    Q2_ALVO_PARES = 2000
    Q2_EPOCAS = 3
    Q3_EPOCAS = 3
    BONUS_MAX_STEPS = -1
    N_BENCHMARK_Q1 = 25


def tempo_decorrido_h():
    return (time.time() - T0) / 3600


random.seed(SEED)
print(f'MODO_TESTE={MODO_TESTE} | RODAR_Q1={RODAR_Q1} | RODAR_BONUS_15B={RODAR_BONUS_15B}')
print(f'Modelo principal: {MODEL_NAME} | MAX_LEN={MAX_LEN} | SEED={SEED}')

### Instalação segura de dependências

A stack do Kaggle (torch, transformers 5.x, datasets, accelerate) fica intocada: reinstalar qualquer um desses pacotes foi o que quebrou os imports nas primeiras tentativas da Questão 01. Só `peft` e `bitsandbytes` são instalados, com `--no-deps`, e apenas se estiverem ausentes. `peft` ausente é erro fatal imediato (a Questão 03 depende dele); `bitsandbytes` ausente apenas desativa o caminho QLoRA (o LoRA puro cumpre a questão).

In [ ]:
import importlib.util
import subprocess
import sys


def garantir_pacote(nome):
    if importlib.util.find_spec(nome) is None:
        print(f'Instalando {nome} com --no-deps...')
        resultado = subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', nome],
            capture_output=True, text=True)
        if resultado.returncode != 0:
            print(resultado.stderr[-800:])
        importlib.invalidate_caches()


garantir_pacote('peft')
garantir_pacote('bitsandbytes')

PEFT_OK = False
BNB_OK = False
try:
    from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
    import peft
    PEFT_OK = True
except Exception as erro:
    print(f'peft indisponível: {erro}')
try:
    import bitsandbytes
    BNB_OK = True
except Exception as erro:
    print(f'bitsandbytes indisponível: {erro}')

if not PEFT_OK:
    raise RuntimeError('peft é obrigatório para a Questão 3 e não pôde ser carregado. Verifique se a Internet do notebook está ligada.')

import torch
import transformers

print(f'torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__}')
print(f'PEFT_OK={PEFT_OK} | BNB_OK={BNB_OK}')

In [ ]:
import gc
import json
import math
import re
import shutil
import hashlib
import traceback
import unicodedata
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
from datasets import load_dataset, concatenate_datasets
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    TrainerCallback,
    DataCollatorForLanguageModeling,
    default_data_collator,
)

try:
    from transformers import BitsAndBytesConfig
except Exception as erro:
    print(f'BitsAndBytesConfig indisponível: {erro}')
    BNB_OK = False

np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')
if DEVICE.type == 'cuda':
    assert torch.cuda.device_count() == 1, 'Esperada exatamente 1 GPU visível'
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM total: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

print(f'Disco livre: {shutil.disk_usage(".").free / 1e9:.1f} GB')

### Funções compartilhadas

Definidas uma única vez e usadas pelas três questões: salvamento de artefatos, estilo e helpers de gráfico, montagem de prompts no formato Alpaca e limpeza de memória entre seções.

In [ ]:
DIR_FIGURAS = 'figuras'
os.makedirs(DIR_FIGURAS, exist_ok=True)

COR_ANTES = '#b5651d'
COR_DEPOIS = '#12805a'
C_FULL = '#b5651d'
C_LORA = '#2a78d6'
C_QLORA = '#12805a'
C_BONUS = '#4a3aa7'
C_BASE = '#898781'
C_EDA = '#2a78d6'
COR_GRADE = '#e1e0d9'

plt.rcParams.update({
    'font.size': 11,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'figure.facecolor': 'white',
    'axes.grid': False,
    'axes.axisbelow': True,
})

RESULTADOS = {}


def salvar_json(objeto, caminho):
    conteudo = json.dumps(objeto, ensure_ascii=False, indent=2, default=str)
    with open(caminho, 'w', encoding='utf-8') as arquivo:
        arquivo.write(conteudo)
    print(f'Salvo: {caminho}')
    if len(conteudo) < 100000:
        print(f'----- backup em log: inicio de {caminho} -----')
        print(conteudo)
        print(f'----- backup em log: fim de {caminho} -----')


def salvar_fig(fig, nome):
    caminho = os.path.join(DIR_FIGURAS, nome)
    fig.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.show()
    plt.close(fig)
    print(f'Figura salva: {caminho}')


def rotular_barras(ax, barras, sufixo='', casas=2):
    for barra in barras:
        altura = barra.get_height()
        ax.annotate(f'{altura:.{casas}f}{sufixo}',
                    xy=(barra.get_x() + barra.get_width() / 2, altura),
                    xytext=(0, 3), textcoords='offset points',
                    ha='center', va='bottom', fontsize=9, fontweight='bold')


def grafico_metricas_multi(series, titulo, nome_arquivo):
    paineis = [('Entropia Cruzada', 'entropia_cruzada', '', 3),
               ('Perplexidade', 'perplexidade', '', 2),
               ('Acurácia de Tokens (%)', 'acuracia_tokens', '%', 1)]
    largura = 12 + max(0, len(series) - 2) * 1.4
    fig, eixos = plt.subplots(1, 3, figsize=(largura, 4.2))
    for ax, (nome, chave, sufixo, casas) in zip(eixos, paineis):
        rotulos = [s[0] for s in series]
        valores = [s[1][chave] for s in series]
        cores = [s[2] for s in series]
        barras = ax.bar(rotulos, valores, color=cores, width=0.62)
        rotular_barras(ax, barras, sufixo=sufixo, casas=casas)
        ax.set_title(nome)
        ax.set_ylim(0, max(valores) * 1.28 + 1e-9)
        ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
        if len(series) > 2:
            ax.tick_params(axis='x', labelrotation=20)
    fig.suptitle(titulo, fontweight='bold')
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_barh(itens, titulo, nome_arquivo, cor=None, log_x=False, fmt='{:,.0f}'):
    nomes = [str(item[0]) for item in itens]
    valores = [item[1] for item in itens]
    fig, ax = plt.subplots(figsize=(9.5, max(3, 0.42 * len(itens) + 1.2)))
    barras = ax.barh(nomes[::-1], valores[::-1], color=cor or C_EDA)
    if log_x:
        ax.set_xscale('log')
    for barra in barras:
        largura = barra.get_width()
        ax.annotate(fmt.format(largura),
                    xy=(largura, barra.get_y() + barra.get_height() / 2),
                    xytext=(4, 0), textcoords='offset points',
                    va='center', fontsize=9)
    ax.set_title(titulo)
    ax.grid(axis='x', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_hist_comprimentos(comprimentos, cortes, titulo, nome_arquivo, mediana=None):
    dados = [c for c in comprimentos if c > 0]
    bins = np.logspace(np.log10(max(min(dados), 1)), np.log10(max(dados)), 50)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(dados, bins=bins, color=C_EDA, edgecolor='white', linewidth=0.3)
    ax.set_xscale('log')
    for corte in cortes:
        ax.axvline(corte, color='#8a1d1d', linestyle='--', linewidth=1.2)
    if mediana:
        ax.axvline(mediana, color='#4a3aa7', linestyle=':', linewidth=1.4)
        ax.annotate(f'mediana {mediana:,.0f}', xy=(mediana, ax.get_ylim()[1] * 0.9),
                    xytext=(6, 0), textcoords='offset points', fontsize=9, color='#4a3aa7')
    ax.set_xlabel('Comprimento (caracteres, escala log)')
    ax.set_ylabel('Documentos')
    ax.set_title(titulo)
    ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_curva_loss(log_history, titulo, nome_arquivo):
    pontos_treino = [(h['step'], h['loss']) for h in log_history if 'loss' in h and 'eval_loss' not in h]
    pontos_eval = [(h['step'], h['eval_loss']) for h in log_history if 'eval_loss' in h]
    fig, ax = plt.subplots(figsize=(9, 4.2))
    if pontos_treino:
        xs, ys = zip(*pontos_treino)
        ax.plot(xs, ys, color=C_EDA, linewidth=1.6, label='Loss de treino')
    if pontos_eval:
        xs, ys = zip(*pontos_eval)
        ax.plot(xs, ys, color='#184f95', marker='o', linestyle='--', linewidth=1.2, label='Loss de avaliação')
    ax.set_xlabel('Passo')
    ax.set_ylabel('Loss')
    ax.set_title(titulo)
    ax.legend(frameon=False)
    ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_curvas_loss(series, titulo, nome_arquivo):
    fig, ax = plt.subplots(figsize=(9, 4.2))
    for rotulo, log_history, cor in series:
        pontos = [(h['step'], h['loss']) for h in log_history if 'loss' in h and 'eval_loss' not in h]
        if pontos:
            xs, ys = zip(*pontos)
            ax.plot(xs, ys, color=cor, linewidth=1.6, label=rotulo)
    ax.set_xlabel('Passo')
    ax.set_ylabel('Loss de treino')
    ax.set_title(titulo)
    ax.legend(frameon=False)
    ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def grafico_painel_comparativo(entidades, titulo, nome_arquivo):
    paineis = [('Perplexidade (depois)', 'ppl', '', 2),
               ('Acurácia de Tokens (depois, %)', 'acc', '%', 1),
               ('Tempo de treino (min)', 'tempo_min', '', 1),
               ('Pico de VRAM (GB)', 'vram', '', 2)]
    fig, eixos = plt.subplots(2, 2, figsize=(11, 8))
    for ax, (nome, chave, sufixo, casas) in zip(eixos.flat, paineis):
        rotulos = [e['rotulo'] for e in entidades]
        valores = [e[chave] for e in entidades]
        cores = [e['cor'] for e in entidades]
        barras = ax.bar(rotulos, valores, color=cores, width=0.6)
        rotular_barras(ax, barras, sufixo=sufixo, casas=casas)
        ax.set_title(nome)
        ax.set_ylim(0, max(valores) * 1.28 + 1e-9)
        ax.tick_params(axis='x', labelrotation=15)
        ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.suptitle(titulo, fontweight='bold')
    fig.tight_layout()
    salvar_fig(fig, nome_arquivo)


def montar_prompt(instrucao, entrada=''):
    if entrada and entrada.strip():
        return f'### Instrução:\n{instrucao}\n\n### Entrada:\n{entrada}\n\n### Resposta:\n'
    return f'### Instrução:\n{instrucao}\n\n### Resposta:\n'


def normalizar_texto(texto):
    texto = unicodedata.normalize('NFKD', texto.lower())
    texto = ''.join(c for c in texto if not unicodedata.combining(c))
    return re.sub(r'[^a-z0-9]', '', texto)


def taxa_acerto_por_categoria(resultados):
    por_categoria = {}
    for r in resultados:
        acertou = normalizar_texto(r['resposta_esperada']) in normalizar_texto(r['resposta_gerada'])
        ok, total = por_categoria.get(r['cat'], (0, 0))
        por_categoria[r['cat']] = (ok + (1 if acertou else 0), total + 1)
    return {c: round(ok / total * 100, 1) for c, (ok, total) in por_categoria.items()}


def atualizar_zip_parcial():
    os.makedirs('parciais', exist_ok=True)
    for nome in os.listdir('.'):
        if nome.endswith('.json'):
            shutil.copy(nome, os.path.join('parciais', nome))
    if os.path.isdir(DIR_FIGURAS):
        shutil.copytree(DIR_FIGURAS, os.path.join('parciais', DIR_FIGURAS), dirs_exist_ok=True)
    shutil.make_archive('resultados_parciais', 'zip', 'parciais')
    print('Zip parcial atualizado: resultados_parciais.zip')


print('Helpers de artefatos e gráficos definidos.')

In [ ]:
def carregar_modelo_fp32(nome):
    try:
        modelo = AutoModelForCausalLM.from_pretrained(nome, dtype=torch.float32)
    except TypeError:
        modelo = AutoModelForCausalLM.from_pretrained(nome, torch_dtype=torch.float32)
    return modelo.to(DEVICE)


def montar_training_args(**kwargs):
    for tentativa in range(6):
        try:
            return TrainingArguments(**kwargs)
        except (TypeError, ValueError) as erro:
            mensagem = str(erro)
            removido = None
            for chave in list(kwargs):
                if chave in mensagem:
                    removido = chave
                    kwargs.pop(chave)
                    break
            if removido is None:
                raise
            print(f'Argumento removido por incompatibilidade de versão: {removido}')
    raise RuntimeError('Não foi possível montar TrainingArguments')


def calcular_metricas(modelo, tok, textos, prompts=None, batch_size=8):
    modelo.eval()
    total_loss = 0.0
    total_validos = 0
    total_certos = 0
    for inicio in range(0, len(textos), batch_size):
        lote = textos[inicio:inicio + batch_size]
        enc = tok(lote, return_tensors='pt', max_length=MAX_LEN, truncation=True, padding=True)
        input_ids = enc['input_ids'].to(DEVICE)
        attention_mask = enc['attention_mask'].to(DEVICE)
        labels = input_ids.clone()
        labels[attention_mask == 0] = -100
        if prompts is not None:
            for j, prompt in enumerate(prompts[inicio:inicio + batch_size]):
                n_prompt = len(tok(prompt)['input_ids'])
                labels[j, :min(n_prompt, labels.shape[1])] = -100
        with torch.no_grad():
            saida = modelo(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        mascara_validos = labels[:, 1:] != -100
        n_validos = mascara_validos.sum().item()
        if n_validos == 0:
            continue
        total_loss += saida.loss.item() * n_validos
        total_validos += n_validos
        predicoes = saida.logits.argmax(dim=-1)
        certos = ((predicoes[:, :-1] == input_ids[:, 1:]) & mascara_validos).sum().item()
        total_certos += certos
    entropia = total_loss / max(total_validos, 1)
    return {'entropia_cruzada': round(entropia, 4),
            'perplexidade': round(math.exp(min(entropia, 20)), 2),
            'acuracia_tokens': round(total_certos / max(total_validos, 1) * 100, 2)}


class SFTDatasetMasked(Dataset):
    def __init__(self, pares, tok):
        self.exemplos = []
        for prompt, completo in pares:
            ids_prompt = tok(prompt)['input_ids']
            ids_completo = tok(completo)['input_ids']
            ids_resposta = ids_completo[len(ids_prompt):]
            espaco_resposta = MAX_LEN - len(ids_prompt)
            if espaco_resposta < 2:
                continue
            if len(ids_resposta) > espaco_resposta:
                ids_resposta = ids_resposta[:espaco_resposta - 1] + [tok.eos_token_id]
            input_ids = ids_prompt + ids_resposta
            n_pad = MAX_LEN - len(input_ids)
            attention_mask = [1] * len(input_ids) + [0] * n_pad
            labels = [-100] * len(ids_prompt) + list(ids_resposta) + [-100] * n_pad
            input_ids = input_ids + [tok.pad_token_id] * n_pad
            self.exemplos.append({
                'input_ids': torch.tensor(input_ids),
                'attention_mask': torch.tensor(attention_mask),
                'labels': torch.tensor(labels),
            })

    def __len__(self):
        return len(self.exemplos)

    def __getitem__(self, indice):
        return self.exemplos[indice]


def gerar_resposta(modelo, tok, instrucao, entrada='', max_new_tokens=90):
    modelo.eval()
    prompt = montar_prompt(instrucao, entrada)
    entradas = tok(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tok.pad_token_id or tok.eos_token_id)
    return tok.decode(saida[0][entradas['input_ids'].shape[1]:], skip_special_tokens=True).strip()


def gerar_texto_livre(modelo, tok, prompt, max_new_tokens=80):
    modelo.eval()
    entradas = tok(prompt, return_tensors='pt').to(DEVICE)
    with torch.no_grad():
        saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=True,
                                temperature=0.7, top_p=0.9, repetition_penalty=1.1,
                                pad_token_id=tok.eos_token_id)
    return tok.decode(saida[0], skip_special_tokens=True)


def rodar_benchmark(modelo, tok, itens, max_new_tokens=100):
    modelo.eval()
    resultados = []
    for item in itens:
        entradas = tok(item['pergunta'], return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            saida = modelo.generate(**entradas, max_new_tokens=max_new_tokens, do_sample=False,
                                    pad_token_id=tok.pad_token_id or tok.eos_token_id)
        gerada = tok.decode(saida[0][entradas['input_ids'].shape[1]:], skip_special_tokens=True).strip()
        resultados.append({'id': item['id'], 'cat': item['cat'], 'pergunta': item['pergunta'],
                           'resposta_esperada': item['resposta'], 'resposta_gerada': gerada})
    return resultados


def responder_qualitativa(modelo, tok, itens):
    respostas = []
    for item in itens:
        resposta = gerar_resposta(modelo, tok, item['instruction'], item.get('input', ''))
        respostas.append({'id': item['id'], 'resposta': resposta})
    return respostas


def liberar_gpu(*nomes):
    espaco = globals()
    for nome in nomes:
        espaco.pop(nome, None)
    gc.collect()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        print(f'VRAM alocada: {torch.cuda.memory_allocated() / 1e9:.2f} GB | reservada: {torch.cuda.memory_reserved() / 1e9:.2f} GB')


def status_recursos(rotulo):
    print(f'[{rotulo}] tempo decorrido: {tempo_decorrido_h():.2f}h | disco livre: {shutil.disk_usage(".").free / 1e9:.1f} GB')
    if torch.cuda.is_available():
        print(f'[{rotulo}] VRAM alocada: {torch.cuda.memory_allocated() / 1e9:.2f} GB')


def remover_checkpoints(diretorio):
    if os.path.isdir(diretorio):
        shutil.rmtree(diretorio, ignore_errors=True)
        print(f'Checkpoints removidos: {diretorio}')


print('Helpers de modelo, métricas e memória definidos.')

### Pré-voo

Valida em poucos minutos tudo que poderia falhar depois de horas de treino: acesso aos dois datasets do Hugging Face com os campos esperados, tokenizer, GPU única e disco. Se algo estiver errado, o notebook morre aqui, no minuto 3, e não na hora 6.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f'Tokenizer: {MODEL_NAME} | vocabulário {tokenizer.vocab_size:,}')

verificacao_docentes = load_dataset('vickminari/docentesDC', split='train')
assert 'text' in verificacao_docentes.column_names, 'Campo text ausente no docentesDC'
assert 'nome_professor' in verificacao_docentes.column_names, 'Campo nome_professor ausente no docentesDC'
print(f'docentesDC ok: {len(verificacao_docentes):,} linhas | colunas {verificacao_docentes.column_names}')
del verificacao_docentes

verificacao_dompi = load_dataset('gutoportelaa/DOMPI-2025', name='raw', split='entre_rios')
assert 'texto' in verificacao_dompi.column_names, 'Campo texto ausente no DOMPI-2025'
print(f'DOMPI-2025 ok (split entre_rios): {len(verificacao_dompi):,} documentos')
del verificacao_dompi
gc.collect()

livre_gb = shutil.disk_usage('.').free / 1e9
if os.path.isdir('/kaggle'):
    assert livre_gb > 15, f'Disco livre insuficiente: {livre_gb:.1f} GB'

status_recursos('pré-voo concluído')

## Seção 1: Questão 01, pré-treinamento continuado (DOMPI-2025)

Pré-treinamento continuado do Qwen2.5-0.5B no corpus completo do DOM-PI 2025 (77 mil documentos OCR de 12 territórios do Piauí). A configuração de treino é a mesma que já rodou com sucesso nesta mesma GPU (2 épocas, batch efetivo 32, LR 5e-5 cosine, fp32 com autocast fp16, gradient checkpointing). As novidades em relação à rodada anterior: o benchmark de 25 perguntas agora é respondido também pelo modelo treinado (a comparação antes/depois fica salva em JSON), a acurácia de tokens usa o denominador correto (posições deslocadas válidas) e cada etapa gera gráficos.

In [ ]:
if RODAR_Q1:
    TERRITORIOS = ['carnaubais', 'tabuleiros_alto_parnaiba', 'planice_litoran', 'entre_rios',
                   'vale_do_sambito', 'vale_dos_rios_piaui_e_itaueiras', 'cocais', 'mangabeiras',
                   'serra_da_capivara', 'vale_do_rio_guaribas', 'chapada_vale_do_rio_itaim', 'vale_do_caninde']
    splits_dompi = []
    for territorio in TERRITORIOS:
        print(f'Carregando split {territorio}...')
        splits_dompi.append(load_dataset('gutoportelaa/DOMPI-2025', name='raw', split=territorio))
    dataset_dompi = concatenate_datasets(splits_dompi)
    print(f'Total de documentos: {len(dataset_dompi):,}')
    contagem_territorios = Counter(dataset_dompi['territorio'])
    grafico_barh(contagem_territorios.most_common(), 'DOMPI-2025: documentos por território',
                 'q1_eda_territorios.png')
else:
    print('Questão 01 desativada (RODAR_Q1=False)')

In [ ]:
if RODAR_Q1:
    def limpar_texto(texto):
        if not texto:
            return ''
        texto = re.sub(r'\n{3,}', '\n\n', texto)
        texto = re.sub(r' {2,}', ' ', texto)
        texto = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', texto)
        return texto.strip()

    MIN_CHARS_Q1 = 200
    MAX_CHARS_Q1 = 50000
    textos_brutos_q1 = dataset_dompi['texto']
    comprimentos_q1 = [len(t) if t else 0 for t in textos_brutos_q1]
    grafico_hist_comprimentos(comprimentos_q1, [MIN_CHARS_Q1, MAX_CHARS_Q1],
                              'DOMPI-2025: distribuição de comprimento dos documentos',
                              'q1_eda_comprimentos.png')

    textos_q1 = [limpar_texto(t)[:2000] for t in textos_brutos_q1
                 if t and MIN_CHARS_Q1 <= len(t) <= MAX_CHARS_Q1]
    random.seed(SEED)
    random.shuffle(textos_q1)
    if Q1_MAX_DOCS is not None:
        textos_q1 = textos_q1[:Q1_MAX_DOCS + Q1_N_AVALIACAO]
    q1_textos_eval = textos_q1[:Q1_N_AVALIACAO]
    q1_textos_treino = textos_q1[Q1_N_AVALIACAO:]
    print(f'Após filtragem: {len(textos_q1):,} | Treino: {len(q1_textos_treino):,} | Avaliação: {len(q1_textos_eval):,}')
    del textos_brutos_q1, comprimentos_q1, textos_q1, dataset_dompi, splits_dompi
    gc.collect()
else:
    print('Questão 01 desativada')

### Benchmark de 25 perguntas e avaliação do modelo base

O benchmark cobre entidades exatas (CNPJ, CEP, e-mail), atos administrativos, informações territoriais, processos e legislação. As respostas são geradas de forma determinística (greedy) para a comparação antes/depois ser reprodutível. O resultado do modelo base é salvo imediatamente em `resultados_benchmark_base.json`.

In [ ]:
if RODAR_Q1:
    benchmark_q1 = [
        {'id': 1, 'cat': 'Entidades', 'pergunta': 'Qual é o CNPJ da Prefeitura Municipal de Aroazes?', 'resposta': '06.554.984/0001-39'},
        {'id': 2, 'cat': 'Entidades', 'pergunta': 'Qual é o CEP da Prefeitura Municipal de Aroazes?', 'resposta': '64.310-000'},
        {'id': 3, 'cat': 'Entidades', 'pergunta': 'Qual é o endereço da Prefeitura Municipal de Aroazes?', 'resposta': 'Av. 27 de Fevereiro, 691, Centro'},
        {'id': 4, 'cat': 'Entidades', 'pergunta': 'Qual é o CNPJ da Câmara Municipal de Juazeiro do Piauí?', 'resposta': '01.878.514/0001-07'},
        {'id': 5, 'cat': 'Entidades', 'pergunta': 'Qual é o CNPJ da Prefeitura Municipal de Assunção do Piauí?', 'resposta': '01.612.561/0001-04'},
        {'id': 6, 'cat': 'Atos Adm.', 'pergunta': 'Quais são os principais tipos de atos publicados no DOM-PI?', 'resposta': 'Decreto, Portaria, Licitação, Edital, Lei, Ata, LRF'},
        {'id': 7, 'cat': 'Atos Adm.', 'pergunta': 'O que é um Relatório de Gestão Fiscal (RGF) e qual lei o regulamenta?', 'resposta': 'Documento obrigatório de situação financeira regulamentado pela LRF (art. 55 da LC 101/2000)'},
        {'id': 8, 'cat': 'Atos Adm.', 'pergunta': 'O que é uma Dispensa Eletrônica no contexto de licitações?', 'resposta': 'Contratação direta sem licitação realizada por meio eletrônico para aquisições de menor valor'},
        {'id': 9, 'cat': 'Atos Adm.', 'pergunta': 'O que é um Termo de Rescisão/Distrato de Contrato Administrativo?', 'resposta': 'Documento que formaliza o encerramento antecipado e amigável de um contrato administrativo'},
        {'id': 10, 'cat': 'Atos Adm.', 'pergunta': 'O que é uma Ata de Registro de Preços?', 'resposta': 'Documento que formaliza preços registrados após licitação, permitindo compras futuras sem nova licitação'},
        {'id': 11, 'cat': 'Territorial', 'pergunta': 'Quantos territórios compõem o dataset DOMPI-2025?', 'resposta': '12 territórios'},
        {'id': 12, 'cat': 'Territorial', 'pergunta': 'Qual território possui o maior número de documentos no DOMPI-2025?', 'resposta': 'Cocais, com 12.188 documentos'},
        {'id': 13, 'cat': 'Territorial', 'pergunta': 'Em qual cidade são publicados os Diários Oficiais dos Municípios do Piauí?', 'resposta': 'Teresina (PI)'},
        {'id': 14, 'cat': 'Territorial', 'pergunta': 'Qual é o CEP da Prefeitura Municipal de Anísio de Abreu?', 'resposta': '64.780-000'},
        {'id': 15, 'cat': 'Territorial', 'pergunta': 'Qual é o e-mail da Prefeitura Municipal de Anísio de Abreu?', 'resposta': 'pmanisiodeabreupi@gmail.com'},
        {'id': 16, 'cat': 'Processos', 'pergunta': 'O que é o Pregão Eletrônico no contexto das licitações municipais?', 'resposta': 'Modalidade licitatória realizada eletronicamente em que fornecedores competem por lances'},
        {'id': 17, 'cat': 'Processos', 'pergunta': 'O que é o Demonstrativo da Dívida Consolidada no RGF?', 'resposta': 'RGF Anexo 2: apresenta o total das obrigações financeiras do município conforme a LRF'},
        {'id': 18, 'cat': 'Processos', 'pergunta': 'Quais motivos constituem causa para extinção de contrato administrativo?', 'resposta': 'Descumprimento de cláusulas contratuais, irregularidade fiscal, entre outros (Art. 137 da Lei 14.133/2021)'},
        {'id': 19, 'cat': 'Processos', 'pergunta': 'O que é a Comissão Permanente de Licitação?', 'resposta': 'Órgão colegiado da prefeitura responsável por conduzir e fiscalizar processos licitatórios'},
        {'id': 20, 'cat': 'Processos', 'pergunta': 'O que é uma Errata em publicação de diário oficial?', 'resposta': 'Publicação que corrige erro material contido em publicação anterior'},
        {'id': 21, 'cat': 'Legislação', 'pergunta': 'Qual lei federal regulamenta as licitações e contratos municipais atualmente?', 'resposta': 'Lei Federal nº 14.133/2021 (Nova Lei de Licitações), que substituiu a Lei nº 8.666/1993'},
        {'id': 22, 'cat': 'Legislação', 'pergunta': 'O que significa a sigla LRF?', 'resposta': 'Lei de Responsabilidade Fiscal (Lei Complementar nº 101/2000)'},
        {'id': 23, 'cat': 'Legislação', 'pergunta': 'O que é o Orçamento Fiscal e da Seguridade Social na LOA?', 'resposta': 'Os dois orçamentos da LOA: fiscal (governo geral) e seguridade social (saúde, previdência, assistência)'},
        {'id': 24, 'cat': 'Dataset', 'pergunta': 'Quais ferramentas foram usadas para extrair texto dos PDFs no DOMPI-2025?', 'resposta': 'PaddleOCR-GPU (portarias/decretos), Docling (tabelas/relatórios) e PyMuPDF (texto nativo)'},
        {'id': 25, 'cat': 'Dataset', 'pergunta': 'Qual é o volume total de documentos e o período coberto pelo DOMPI-2025?', 'resposta': '77.337 documentos cobrindo publicações de 2025 do DOM-PI'},
    ]
    benchmark_q1 = benchmark_q1[:N_BENCHMARK_Q1]
    salvar_json(benchmark_q1, 'benchmark_dompi25.json')

    model_base = carregar_modelo_fp32(MODEL_NAME)
    n_parametros = sum(p.numel() for p in model_base.parameters())
    print(f'Modelo base carregado: {MODEL_NAME} ({n_parametros / 1e6:.1f}M parâmetros)')

    print(f'Rodando benchmark de {len(benchmark_q1)} perguntas no modelo BASE...')
    q1_benchmark_antes = rodar_benchmark(model_base, tokenizer, benchmark_q1)
    salvar_json(q1_benchmark_antes, 'resultados_benchmark_base.json')
    for r in q1_benchmark_antes[:5]:
        print(f"[{r['id']}] {r['pergunta']}")
        print(f"    esperada: {r['resposta_esperada']}")
        print(f"    gerada  : {r['resposta_gerada'][:90]}")

In [ ]:
if RODAR_Q1:
    print('Calculando métricas do modelo base no conjunto de avaliação...')
    q1_metricas_antes = calcular_metricas(model_base, tokenizer, q1_textos_eval)
    print(f'ANTES do pré-treino: {q1_metricas_antes}')

    q1_prompts_livres = ['O Prefeito Municipal, no uso de suas atribuições legais,',
                         'PORTARIA Nº 001/2025. Nomeia servidor para o cargo de',
                         'EXTRATO DE CONTRATO. Contratante: Prefeitura Municipal de']
    torch.manual_seed(SEED)
    q1_geracoes_antes = {}
    for prompt in q1_prompts_livres:
        q1_geracoes_antes[prompt] = gerar_texto_livre(model_base, tokenizer, prompt)
        print(f'PROMPT: {prompt}')
        print(f'BASE  : {q1_geracoes_antes[prompt]}')
        print('-' * 80)
    liberar_gpu('model_base')

### Treinamento

O modelo de treino é carregado fresco em fp32 (o treino usa `fp16=True`, que exige pesos mestres em fp32; sem isso o transformers 5 carrega o Qwen em bf16 e o GradScaler quebra na T4). `save_only_model=True` e `save_total_limit=1` mantêm o disco sob controle (o checkpoint fp32 completo com optimizer ocuparia ~5,7GB) e `load_best_model_at_end` garante que a avaliação final usa o melhor checkpoint por `eval_loss`.

In [ ]:
if RODAR_Q1:
    class DiarioDataset(Dataset):
        def __init__(self, textos, tok):
            self.enc = tok(textos, max_length=MAX_LEN, truncation=True)

        def __len__(self):
            return len(self.enc['input_ids'])

        def __getitem__(self, indice):
            return {'input_ids': self.enc['input_ids'][indice]}

    print('Tokenizando dados de treino...')
    q1_train_ds = DiarioDataset(q1_textos_treino, tokenizer)
    print('Tokenizando dados de avaliação...')
    q1_eval_ds = DiarioDataset(q1_textos_eval, tokenizer)
    collator_q1 = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
    print(f'Treino: {len(q1_train_ds):,} | Avaliação: {len(q1_eval_ds):,}')

In [ ]:
if RODAR_Q1:
    model_ft = carregar_modelo_fp32(MODEL_NAME)
    model_ft.config.use_cache = False
    OUTPUT_DIR_Q1 = 'dompi_qwen25'
    args_q1 = montar_training_args(
        output_dir=OUTPUT_DIR_Q1,
        num_train_epochs=Q1_EPOCAS,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={'use_reentrant': False},
        learning_rate=5e-5,
        weight_decay=0.01,
        warmup_steps=0.05,
        lr_scheduler_type='cosine',
        eval_strategy='epoch',
        save_strategy='epoch',
        save_total_limit=1,
        save_only_model=True,
        load_best_model_at_end=True,
        metric_for_best_model='eval_loss',
        fp16=(DEVICE.type == 'cuda'),
        dataloader_num_workers=2,
        logging_steps=100,
        disable_tqdm=True,
        report_to='none',
    )
    if DEVICE.type == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    trainer_q1 = Trainer(model=model_ft, args=args_q1, train_dataset=q1_train_ds,
                         eval_dataset=q1_eval_ds, data_collator=collator_q1)
    print(f'Iniciando pré-treinamento continuado ({Q1_EPOCAS} épocas, batch efetivo 32)...')
    resultado_treino_q1 = trainer_q1.train()
    q1_tempo_treino = resultado_treino_q1.metrics['train_runtime']
    q1_vram_pico = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == 'cuda' else 0.0
    q1_log_history = [dict(h) for h in trainer_q1.state.log_history]
    q1_loss_final = resultado_treino_q1.training_loss
    print(f'Treino concluído: loss final {q1_loss_final:.4f} | tempo {q1_tempo_treino:.0f}s | pico de VRAM {q1_vram_pico:.2f} GB')

In [ ]:
if RODAR_Q1:
    grafico_curva_loss(q1_log_history, 'Questão 1: curva de loss do pré-treino continuado', 'q1_curva_loss.png')
    print('Calculando métricas do modelo treinado (melhor checkpoint, já recarregado pelo Trainer)...')
    q1_metricas_depois = calcular_metricas(model_ft, tokenizer, q1_textos_eval)
    print(f'DEPOIS do pré-treino: {q1_metricas_depois}')
    grafico_metricas_multi(
        [('Antes', q1_metricas_antes, COR_ANTES), ('Depois', q1_metricas_depois, COR_DEPOIS)],
        'Questão 1: métricas antes e depois do pré-treino continuado', 'q1_metricas.png')

### Comparação antes/depois: geração livre e benchmark

A lacuna da rodada anterior é fechada aqui: o mesmo benchmark de 25 perguntas é respondido pelo modelo treinado com os mesmos parâmetros de geração, e a comparação fica salva em `resultados_benchmark_treinado.json`. O gráfico usa uma métrica frouxa (a resposta de referência normalizada está contida na resposta gerada), útil como visão geral; a leitura fina é qualitativa, na tabela impressa.

In [ ]:
if RODAR_Q1:
    torch.manual_seed(SEED)
    q1_geracoes_depois = {}
    for prompt in q1_prompts_livres:
        q1_geracoes_depois[prompt] = gerar_texto_livre(model_ft, tokenizer, prompt)
        print(f'PROMPT : {prompt}')
        print(f'ANTES  : {q1_geracoes_antes[prompt]}')
        print(f'DEPOIS : {q1_geracoes_depois[prompt]}')
        print('-' * 80)

    print(f'Rodando benchmark de {len(benchmark_q1)} perguntas no modelo TREINADO...')
    q1_benchmark_depois = rodar_benchmark(model_ft, tokenizer, benchmark_q1)
    salvar_json(q1_benchmark_depois, 'resultados_benchmark_treinado.json')

    for antes, depois in zip(q1_benchmark_antes, q1_benchmark_depois):
        print(f"[{antes['id']}][{antes['cat']}] {antes['pergunta']}")
        print(f"    esperada: {antes['resposta_esperada']}")
        print(f"    base    : {antes['resposta_gerada'][:100]}")
        print(f"    treinado: {depois['resposta_gerada'][:100]}")

    taxas_antes = taxa_acerto_por_categoria(q1_benchmark_antes)
    taxas_depois = taxa_acerto_por_categoria(q1_benchmark_depois)
    categorias = sorted(set(list(taxas_antes) + list(taxas_depois)))
    fig, ax = plt.subplots(figsize=(9, 4.2))
    posicoes = np.arange(len(categorias))
    ax.bar(posicoes - 0.19, [taxas_antes.get(c, 0) for c in categorias], width=0.38, color=COR_ANTES, label='Antes')
    ax.bar(posicoes + 0.19, [taxas_depois.get(c, 0) for c in categorias], width=0.38, color=COR_DEPOIS, label='Depois')
    ax.set_xticks(posicoes)
    ax.set_xticklabels(categorias)
    ax.set_ylabel('Respostas contendo a referência (%)')
    ax.set_title('Questão 1: benchmark de QA por categoria (métrica frouxa de substring)')
    ax.legend(frameon=False)
    ax.grid(axis='y', color=COR_GRADE, linewidth=0.8)
    fig.tight_layout()
    salvar_fig(fig, 'q1_benchmark_qa.png')

In [ ]:
if RODAR_Q1:
    resultados_metricas_q1 = {
        'modelo': MODEL_NAME,
        'dataset': 'gutoportelaa/DOMPI-2025',
        'n_treino': len(q1_textos_treino),
        'n_avaliacao': len(q1_textos_eval),
        'hiperparametros': {'epocas': Q1_EPOCAS, 'micro_batch': 4, 'grad_accum': 8,
                            'batch_efetivo': 32, 'learning_rate': 5e-05, 'max_length': MAX_LEN},
        'metricas_antes': q1_metricas_antes,
        'metricas_depois': q1_metricas_depois,
        'delta': {
            'entropia_cruzada': round(q1_metricas_depois['entropia_cruzada'] - q1_metricas_antes['entropia_cruzada'], 4),
            'perplexidade': round(q1_metricas_depois['perplexidade'] - q1_metricas_antes['perplexidade'], 2),
            'acuracia_tokens_pp': round(q1_metricas_depois['acuracia_tokens'] - q1_metricas_antes['acuracia_tokens'], 2),
        },
        'treino': {'tempo_s': round(q1_tempo_treino, 1), 'vram_pico_gb': round(q1_vram_pico, 2),
                   'loss_final': round(q1_loss_final, 4)},
        'geracoes_livres': {'antes': q1_geracoes_antes, 'depois': q1_geracoes_depois},
        'log_history': q1_log_history,
    }
    salvar_json(resultados_metricas_q1, 'resultados_metricas.json')
    RESULTADOS['q1'] = resultados_metricas_q1

    model_ft.half()
    model_ft.save_pretrained('q1_modelo_final')
    tokenizer.save_pretrained('q1_modelo_final')
    print('Modelo da Questão 1 salvo em fp16 em q1_modelo_final/')

    remover_checkpoints(OUTPUT_DIR_Q1)
    liberar_gpu('model_ft', 'trainer_q1', 'q1_train_ds', 'q1_eval_ds', 'collator_q1', 'q1_textos_treino')
    atualizar_zip_parcial()
    status_recursos('fim da Questão 1')